# 02 — Late Fusion Model

Implement a dual-branch network: temporal branch + sentiment branch.

In [ ]:
import torch
from torch import nn

torch.manual_seed(8)

In [ ]:
class ContextQuantFusionNet(nn.Module):
    def __init__(self, price_features=2, text_features=2, hidden=32, n_classes=5):
        super().__init__()
        self.temporal = nn.LSTM(price_features, hidden, batch_first=True)
        self.text_branch = nn.Sequential(
            nn.Linear(text_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, price_seq, text_vec):
        temporal_out, _ = self.temporal(price_seq)
        temporal_latent = temporal_out[:, -1, :]
        text_latent = self.text_branch(text_vec)
        fused = torch.cat([temporal_latent, text_latent], dim=1)
        return self.fusion(fused)

model = ContextQuantFusionNet()
print(model)

In [ ]:
batch = 32
price_seq = torch.randn(batch, 20, 2)
text_vec = torch.randn(batch, 2)
logits = model(price_seq, text_vec)
print('Output shape:', logits.shape)  # [batch, 5]

## Exercises
1. Increase temporal branch capacity with 2 LSTM layers.
2. Add dropout to the fusion head and compare stability.